# Retrain T1.5 + H-R3 margin filter ablation

Tests the H-R3 hard-negative margin filter on top of the validated
T1.5 recipe. Four arms, same stores + eval set + seed:

| Arm | `margin_min` | `margin_max` | Intent |
|---|---|---|---|
| 0 (baseline) | `None` | `None` | T1.5 as-is (2026-04-19 Colab: +5.00% macro) |
| A            | `0.05` | `None` | drop ambiguous-close negs only |
| B            | `0.05` | `0.30` | full band (default proposal) |
| C            | `0.10` | `0.35` | tighter band |

Target: close the NFCorpus gap (baseline +2.22% vs v5's +18.3%)
without regressing SciFact (+5.41%) or FiQA (+7.37%).

Each arm = one `retrain_multi` call (~25-35 min on T4). Run the
cells you want; the summary cell at the end reads whichever arms
completed and produces a comparison table.

In [ ]:
# Cell 1: Setup -- clone the H-R3 branch.
!pip install -q 'sentence-transformers>=3' torch 'accelerate>=1.1.0'
!rm -rf /content/vstash
!git clone --branch feat/retrain-hr3-margin-filter https://github.com/stffns/vstash.git /content/vstash
%cd /content/vstash
!pip install -q -e .

In [ ]:
# Cell 2: Download BEIR + ingest into per-dataset stores.
import os
import shutil
import sys

os.chdir("/content/vstash")
sys.path.insert(0, "/content/vstash")
os.makedirs("experiments/data", exist_ok=True)

from experiments.beir_benchmark import download_beir, load_beir  # noqa: E402
from sentence_transformers import SentenceTransformer  # noqa: E402
from vstash.store import VstashStore  # noqa: E402

BASE_MODEL = "BAAI/bge-small-en-v1.5"
DATASETS = ["scifact", "nfcorpus", "fiqa"]
store_paths = {name: f"/tmp/retrain_hr3_{name}.db" for name in DATASETS}

# Clean any stale stores.
for p in store_paths.values():
    for suffix in ("", "-wal", "-shm"):
        target = p + suffix
        if os.path.isfile(target):
            os.remove(target)

model = SentenceTransformer(BASE_MODEL)

per_dataset = {}
stores = {}
for name in DATASETS:
    cache = download_beir(name)
    corpus, queries, qrels = load_beir(cache)
    doc_ids = list(corpus.keys())
    print(f"[{name}] corpus: {len(doc_ids)} docs | queries: {len(queries)} | qrels: {len(qrels)}")
    per_dataset[name] = {"corpus": corpus, "queries": queries, "qrels": qrels}

    texts = [(corpus[d].get("title", "") + " " + corpus[d].get("text", "")).strip() for d in doc_ids]
    vecs = model.encode(texts, normalize_embeddings=True, show_progress_bar=True, batch_size=128)
    store = VstashStore(store_paths[name], embedding_dim=int(vecs.shape[1]))
    for doc_id, text, vec in zip(doc_ids, texts, vecs):
        store.add_document(
            path=f"{name}://{doc_id}",
            title=corpus[doc_id].get("title", "")[:80] or doc_id,
            chunks=[text],
            embeddings=[list(map(float, vec))],
        )
    stats = store.stats()
    print(f"  -> ingested {stats.documents} docs / {stats.chunks} chunks")
    stores[name] = store

In [ ]:
# Cell 3: Build per-dataset eval sets from real qrels. Reused as
# training_queries_by_dataset too (v5 recipe).
from vstash.retrain import qrels_to_eval_queries

eval_queries_by_dataset = {}
for name, bundle in per_dataset.items():
    eqs = qrels_to_eval_queries(
        queries=bundle["queries"],
        qrels=bundle["qrels"],
        path_for_doc_id=lambda doc_id, d=name: f"{d}://{doc_id}",
    )
    eval_queries_by_dataset[name] = eqs
    print(f"[{name}] eval_queries (real qrels): {len(eqs)}")

In [ ]:
# Cell 4: Shared retrain_multi config. Arm cells below only override
# margin_min / margin_max / output_path so results are cleanly isolated.
import time
from vstash.retrain import retrain_multi

TOTAL_TRIPLES = 30000
SAMPLING = "temperature"
TEMPERATURE = 0.5
EVAL_NOISE = max(max(s.stats().chunks for s in stores.values()), 10000)
SEED = 42


def run_arm(
    arm_name: str,
    margin_min: float | None,
    margin_max: float | None,
):
    output_path = f"/content/retrained_hr3_{arm_name}"
    for p in (output_path, output_path + ".candidate", output_path + ".old"):
        if os.path.isdir(p):
            shutil.rmtree(p)
    t0 = time.perf_counter()
    result = retrain_multi(
        stores,
        base_model=BASE_MODEL,
        output_path=output_path,
        sampling=SAMPLING,
        temperature=TEMPERATURE,
        total_triples=TOTAL_TRIPLES,
        epochs=2,
        lr=3e-6,
        batch_size=32,
        use_amp=True,
        max_seq_length=256,
        bulk_mine=True,
        bulk_eval=True,
        training_queries_by_dataset=eval_queries_by_dataset,
        eval_queries_by_dataset=eval_queries_by_dataset,
        eval_noise_size=EVAL_NOISE,
        min_gain=-1.0,  # never hard-fail; we want every arm's meta.json
        per_dataset_gate=False,
        seed=SEED,
        margin_min=margin_min,
        margin_max=margin_max,
    )
    elapsed = time.perf_counter() - t0
    print(
        f"\n[{arm_name}] margin=[{margin_min}, {margin_max}]  "
        f"macro delta NDCG@10={result.macro_delta_ndcg:+.4f} ({result.macro_delta_ndcg * 100:+.2f}%)  "
        f"total_pairs={result.total_pairs}  elapsed={elapsed:.1f}s"
    )
    return result, output_path

## Arms

Each arm = ~25-35 min on T4. Run the ones you want; the summary
cell below picks up whichever completed.

In [ ]:
# Arm 0: baseline (no filter). Reproduces the 2026-04-19 Colab result:
# +5.00% macro (SciFact +5.41%, NFCorpus +2.22%, FiQA +7.37%).
result_baseline, path_baseline = run_arm("baseline", margin_min=None, margin_max=None)

In [ ]:
# Arm A: drop too-close negs only. Hypothesis: NFCorpus's 50-gold-
# per-query load includes many near-duplicate relevant docs that the
# labeled miner surfaces as hard negs; cutting margin<0.05 removes them.
result_a, path_a = run_arm("arm_a", margin_min=0.05, margin_max=None)

In [ ]:
# Arm B: full band [0.05, 0.30]. Hypothesis in hypotheses.md's H-R3
# default: drop both tails (ambiguous + easy).
result_b, path_b = run_arm("arm_b", margin_min=0.05, margin_max=0.30)

In [ ]:
# Arm C: tighter band [0.10, 0.35]. Higher lower bound removes more
# ambiguous negs; higher upper bound keeps a few easier ones to
# retain gradient signal on smaller corpora.
result_c, path_c = run_arm("arm_c", margin_min=0.10, margin_max=0.35)

In [ ]:
# Summary: read every arm's training_meta.json and build a comparison
# table. Cells that did not run leave their meta absent and are
# skipped automatically.
import json
from pathlib import Path

arms = [
    ("baseline", path_baseline if "path_baseline" in dir() else None, None, None),
    ("arm_a", path_a if "path_a" in dir() else None, 0.05, None),
    ("arm_b", path_b if "path_b" in dir() else None, 0.05, 0.30),
    ("arm_c", path_c if "path_c" in dir() else None, 0.10, 0.35),
]


def _load(path):
    if not path:
        return None
    # retrain_multi saves the promoted model path; meta lives in the
    # promoted dir after a successful run, or in the .candidate dir if
    # the gate rejected. We pass min_gain=-1 so everything promotes.
    for candidate in (Path(path), Path(path + ".candidate"), Path(path + ".old")):
        meta_path = candidate / "training_meta.json"
        if meta_path.exists():
            return json.loads(meta_path.read_text())
    return None


print(
    f"{'arm':<10} {'margin':<16} {'pairs':>7}  {'NFCorpus':>10}  {'SciFact':>10}  {'FiQA':>10}  {'macro':>10}"
)
print("-" * 88)
for name, path, mmin, mmax in arms:
    meta = _load(path)
    if meta is None:
        continue
    multi = meta.get("multi_eval", {})
    pairs = sum(multi.get("per_dataset_pairs", {}).values())
    base = multi.get("per_dataset_baseline", {})
    fin = multi.get("per_dataset_final", {})

    def _delta(ds):
        b = base.get(ds, {}).get("ndcg_at_10")
        f = fin.get(ds, {}).get("ndcg_at_10")
        if b is None or f is None:
            return "-"
        return f"{(f - b) * 100:+.2f}%"

    macro_delta = multi.get("macro_delta_ndcg", 0.0) * 100
    margin_str = f"[{mmin}, {mmax}]"
    print(
        f"{name:<10} {margin_str:<16} {pairs:>7}  {_delta('nfcorpus'):>10}  "
        f"{_delta('scifact'):>10}  {_delta('fiqa'):>10}  {macro_delta:>+9.2f}%"
    )

print()
print("Target: NFCorpus > +5% on at least one arm, macro non-regressive vs baseline.")